## Laboratory 13 — cloud variant
### Exercise 2 — Remote CSV / Parquet tools

In [ ]:
import io
import json
import os
import httpx
import polars as pl
from typing import Callable

from dotenv import load_dotenv
from openai import OpenAI


def make_llm_request(prompt: str, dataset_url: str) -> str:
    load_dotenv()
    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data assistant that answers questions about a dataset "
                "by reading it with the available tools. The dataset is "
                f"available at this URL: {dataset_url}"
            ),
        },
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model="gemini-3.5-flash",
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            reasoning_effort="low",
        )
        resp_message = response.choices[0].message
        messages.append(resp_message.model_dump(exclude_none=True))

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [

        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Downloads and reads a CSV file from a specified URL using Polars, returning the content or a summary as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "format": "uri",
                            "description": "The HTTP/HTTPS URL of the CSV file to read."
                        },
                        "row_limit": {
                            "type": "integer",
                            "description": "Optional maximum number of rows to return to prevent overloading the context window.",
                            "default": 100
                        },
                        "has_header": {
                            "type": "boolean",
                            "description": "Whether the CSV file contains a header row.",
                            "default": True
                        }
                    },
                    "required": ["url"]
                }
            }
        },

        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Downloads and reads a Parquet file from a specified URL using Polars, returning the content or a summary as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "format": "uri",
                            "description": "The HTTP/HTTPS URL of the Parquet file to read."
                        },
                        "row_limit": {
                            "type": "integer",
                            "description": "Optional maximum number of rows to return to prevent overloading the context window.",
                            "default": 100
                        },
                        "columns": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            },
                            "description": "Optional list of specific columns to select from the Parquet file."
                        }
                    },
                    "required": ["url"]
                }
            }
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": handle_read_remote_csv,
        "read_remote_parquet": handle_read_remote_parquet,
    }

    return tool_definitions, tool_name_to_callable


def handle_read_remote_csv(url: str, row_limit: int = 100, has_header: bool = True) -> str:
    content = httpx.get(url, follow_redirects=True).raise_for_status().content
    df = pl.read_csv(io.BytesIO(content), has_header=has_header, n_rows=row_limit)
    return str(df)

def handle_read_remote_parquet(url: str, row_limit: int = 100, columns: list[str] | None = None) -> str:
    content = httpx.get(url, follow_redirects=True).raise_for_status().content
    df = pl.read_parquet(io.BytesIO(content), columns=columns)

    if row_limit:
        df = df.head(row_limit)

    return str(df)

csv_url = "https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv"
parquet_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

if __name__ == "__main__":
    # CSV tests (ApisTox dataset)
    prompt = "What is the name for CID=971?"
    response = make_llm_request(prompt, csv_url)
    print("Response:\n", response)

    print()

    prompt = "What is CAS for CID=971?"
    response = make_llm_request(prompt, csv_url)
    print("Response:\n", response)

    print()

    prompt = "What is the CID for Benzoic acid?"
    response = make_llm_request(prompt, csv_url)
    print("Response:\n", response)

    print()

    # Parquet tests (NYC yellow taxi, January 2024)
    prompt = "What columns does this dataset have?"
    response = make_llm_request(prompt, parquet_url)
    print("Response:\n", response)

    print()

    prompt = "What is the most common payment type in the first 100 rows?"
    response = make_llm_request(prompt, parquet_url)
    print("Response:\n", response)

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'a8zfxxst', 'function': {'arguments': '{"url":"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv","row_limit":10}', 'name': 'read_remote_csv'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'EpgDCpUDAQw51seE/KIF7Z8iC2WVRrJNZDWIXtPIqN+d5gqRevTZUTpTennLaDgQm7z+hNhYdsZuYeniFpLKJ9hvLJpBkFTUKIhkkEMW5/FTDvOHQYJYp2jTHX/rhMw8CDw5/95pv1ou0vuhaO432Rpgals63wr2xhUhxiZURqhbYVPXTsvR2wk9R01K2vkTFsNVuRi32Yf+LW/LYN9nBmrMvgSDXD5QmdblyaJncTBbRHdbTQaEt5+g76UkM6HA+tRdwuuaVLmIMsBKsjrjz3ZRkLyHLPoN8SLBpDbllypkvSV6Wzrx7tmkLOpeZt/j7F+7FMlUADqDv1ze0kE0KwI/yFWMW2OCb484k+17Zo/vyKJpOx5ZatuOOSfpdOZsE51GPvhZsKHeCxOvrifwf3jTNGeqLYMcKqnhk+2WN6zXTSgoQol/u3CEnsLCzKDk1uK5YdSSoUpOsow8eZMrpbSJXtSR0LpmAhLykw+VLTM5R3ekurDJF0CenblgrOhhNlN/sj6d1NA9Oyb2zKpcBNOyLwIxlAvh0Nru'}}}]}

Generated message: